# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, datasets may have multiple record sets. Each record set, field, and column is referenced by its unique `@id`. Let's inspect what this dataset offers:

In [ ]:
# List all record sets and their `@id`s and names
print("\nAvailable Record Sets:")
record_sets = [rs for rs in dataset.metadata.recordSets]
for rs in record_sets:
    print(f"  - @id: {rs['@id']} | name: {rs.get('name', '')}")

# Pick the first record set for further exploration
if record_sets:
    record_set_id = record_sets[0]['@id']

    # List all fields for the chosen record set
    print(f"\nFields in record set '{record_set_id}':")
    fields = record_sets[0]['fields']
    for field in fields:
        print(f"  - @id: {field['@id']} | name: {field.get('name', '')} | datatype: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview to extract data. All `mlcroissant` operations should refer to entities by their `@id`. You can inspect field types, choose relevant columns, and peek at the content.

In [ ]:
# Prepare for data extraction
import pprint

# List all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]

# Load all as DataFrames, using the record set @id as key
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Explore the first available record set
focus_record_set_id = record_set_ids[0]
print(f"Fields for record set: {focus_record_set_id}")
print(dataframes[focus_record_set_id].columns.tolist())
# Preview data
dataframes[focus_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations may include removing outliers, transforming distributions, and grouping data by categorical fields. Here, we demonstrate this workflow referencing fields by their `@id`.

In [ ]:
import numpy as np

focus_df = dataframes[focus_record_set_id]

# Identify a numeric field for demonstration. We fetch the first numeric column if available.
numeric_fields = [col for col in focus_df.columns if pd.api.types.is_numeric_dtype(focus_df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    print("No numeric fields found for the selected record set.")
    numeric_field_id = None

# Filtering (example: keep records with value > 10 in the numeric field)
if numeric_field_id:
    threshold = 10
    filtered_df = focus_df[focus_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if any are present
    categorical_fields = [col for col in focus_df.columns if pd.api.types.is_object_dtype(focus_df[col]) and col != numeric_field_id]
    if categorical_fields:
        group_field_id = categorical_fields[0]
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical fields found for grouping.")
else:
    print("Skipping EDA steps due to lack of numeric fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot, for demonstration, a histogram of the first numeric field in the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(focus_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
In this notebook, we loaded the FAIR² Croissant dataset and explored its record sets, fields, and contents using the `mlcroissant` library. We demonstrated how to extract, filter, normalize, and visualize data—all while referencing dataset entities by their Croissant `@id`.

For deeper exploration, consult the FAIR² dataset schema for additional record sets, fields, or to join multiple tables. Ensure all manipulations in your analyses reference data entities by their `@id` for consistency and maintainability.